### Getting Started with Local Large Language Models

In this file, a local language model will be applied to answer easy questions in multibody problems first textually and then from an image. 

First, the libraries are loaded:  
<!--* gpt4all: used to run models locally; can be used both as a GUI and a Python library -->
* transformers: library for pre-trained models for inference applications and fine-tuning
* Hugging Face Hub: is a git-based repository, offering a wide range of pre-trained models
<!-- * Ollama: software for running LLMs locally; Python and Java-script library and rest-API. Supports models in .gguf  format. -->
First we will run the gemma4 2EB model. The _E_ models from this series are developted to run on edge devices, thus are a compromise between inference speed and quality and has 2 Billion parameters. 
If you don't have a GPU locally, the second model - ministral - will not be run locally but instead a previously generated output will be loaded. 

Note that the respective model is automatically downloaded when running this script. You can technically run models that exceed the available memory by swapping to the disk, but this will slow them down considerably. 

Task: 
* Run script locally
* Optional: browse models on [huggingface](https://huggingface.co/) and find a model you want to run. Change the prompt 


In [1]:

from transformers import AutoConfig, pipeline
from huggingface_hub import hf_hub_download
import torch # only used to check if cuda is available
import time
import os
import logging
import warnings
import pickle
from transformers import logging as tlogging

# don't show warnings from transformers
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "true"
tlogging.set_verbosity_error()
warnings.filterwarnings("ignore", module=r"transformers.*")
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)


MODEL_ID = "google/gemma-4-E2B-it" # approx 10GB disk size

AutoConfig.from_pretrained(MODEL_ID)
# from the Config you can read many architectural details like number of attention heads, quantization, activation function, ...

Gemma4Config {
  "architectures": [
    "Gemma4ForConditionalGeneration"
  ],
  "audio_config": {
    "_name_or_path": "",
    "architectures": null,
    "attention_chunk_size": 12,
    "attention_context_left": 13,
    "attention_context_right": 0,
    "attention_invalid_logits_value": -1000000000.0,
    "attention_logit_cap": 50.0,
    "chunk_size_feed_forward": 0,
    "conv_kernel_size": 5,
    "dtype": "bfloat16",
    "gradient_clipping": 10000000000.0,
    "hidden_act": "silu",
    "hidden_size": 1024,
    "id2label": {
      "0": "LABEL_0",
      "1": "LABEL_1"
    },
    "initializer_range": 0.02,
    "is_encoder_decoder": false,
    "label2id": {
      "LABEL_0": 0,
      "LABEL_1": 1
    },
    "model_type": "gemma4_audio",
    "num_attention_heads": 8,
    "num_hidden_layers": 12,
    "output_attentions": false,
    "output_hidden_states": false,
    "output_proj_dims": 1536,
    "problem_type": null,
    "residual_weight": 0.5,
    "return_dict": true,
    "rms_norm_eps": 1e

### Load Model


The model is automatically downloaded and then loaded either on the GPU ('cuda') if available or into the RAM for running inference on the CPU.  
By default, the model location is:  
<!-- <code> `C:\Users\<username>\.cache\huggingface\` </code> -->
<code> `%USERPROFILE%\.cache\huggingface\hub` </code>

__Note__: If you are short on memory on your PC, you should later **delete** the LLM file model manually afterwards as it will be not deleted with the Python environment and the model sizes  usually are multiple GBs. 

In [2]:
# modelPath = hf_hub_download(repo_id=repo_id, filename=filename)
print('model {}: downloading/loading \n'.format(MODEL_ID))
pipe = pipeline(
        task="any-to-any",
        model=MODEL_ID,
        device_map="auto",
        dtype="auto"
)
# this might take a few minutes to load
print(pipe)

model google/gemma-4-E2B-it: downloading/loading 



Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

AnyToAnyPipeline: {'model': 'Gemma4ForConditionalGeneration', 'dtype': 'bfloat16', 'device': 'cpu', 'input_modalities': ('image', 'text', 'video', 'audio'), 'output_modalities': ('text',)}


In [3]:
strQuestion = "How many eigenmodes does a two-mass spring-damper have? Keep the answer short."
myInput = f"<|turn>user\n{strQuestion}\n<|turn>model\n"

print(strQuestion)
t1 = time.time()
output = pipe(myInput, max_new_tokens=512)
dt = time.time() - t1
print(f'inference took {round(dt, 2)}s')


How many eigenmodes does a two-mass spring-damper have? Keep the answer short.
inference took 5.71s


In [4]:
print('The pipeline created a dictionary with keys: ')
print(output[0].keys())
print("generated text:\n", output[0]['generated_text'])

The pipeline created a dictionary with keys: 
dict_keys(['input_text', 'generated_text'])
generated text:
 <|turn>user
How many eigenmodes does a two-mass spring-damper have? Keep the answer short.
<|turn>model
Two<turn|>


#### Prompts and Formatting
Different models and interfaces require different formatting of input. 
This can be found at the modelcard; for gemma4 it can be found [here](https://ai.google.dev/gemma/docs/core/huggingface_inference). 

In [5]:
from PIL import Image 
from transformers import AutoProcessor, Mistral3ForConditionalGeneration 
from IPython.display import display 
import requests
from io import BytesIO

pickle_path = "generated_ids.pkl"
processor = AutoProcessor.from_pretrained("mistralai/Mistral-Small-3.1-24B-Instruct-2503")
promptInput = "What kind of mechanism do you see in the image?"

# set true to run Mistral model - note that this also requires also to download it
flagGeneration = False
if not(flagGeneration) and not(torch.cuda.is_available()):
    print("Loading previously saved output")
    if os.path.exists(pickle_path):
        with open(pickle_path, "rb") as f:
            generate_ids = pickle.load(f)
    else:
        raise FileNotFoundError(f"Could not find {{pickle_path}}. Run once with a GPU to create saved generated_ids.")
else:
    model = Mistral3ForConditionalGeneration.from_pretrained("mistralai/Mistral-Small-3.1-24B-Instruct-2503")
    
    # image_url = "https://raw.githubusercontent.com/jgerstmayr/EXUDYN/master/docs/theDoc/figures/open_closed_loop.png"
    image_url = "https://raw.githubusercontent.com/jgerstmayr/EXUDYN/master/docs/theDoc/figures/pendulumConstraint.png"
    image = Image.open(BytesIO(requests.get(image_url).content)).convert("RGB")
    print('\n'*2, promptInput, '\n')
    display(image)

    messages = [
        {
            "role": "user", "content": [
                {"type": "image", "url": image_url},
                {"type": "text", "text": promptInput},
            ]
        },
    ]

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=True, 
        max_length=4096*4,
    )

    generate_ids = model.generate(**inputs, max_new_tokens=1024)


Loading previously saved output


In [6]:
gen_decode = processor.batch_decode(generate_ids, skip_special_tokens=False)[0]
# print('model input: \n', gen_decode.split(promptInput)[0].split('[IMG]')[0] + '\n<image>\n' + promptInput)
print('model input: \n', gen_decode.split(promptInput)[0])
print('*'*30 + '\nmodel output: \n', gen_decode.split(promptInput)[-1].split('[/INST]')[-1])


model input: 
 <s>[SYSTEM_PROMPT]You are Mistral Small 3, a Large Language Model (LLM) created by Mistral AI, a French startup headquartered in Paris.
Your knowledge base was last updated on 2023-10-01. The current date is 2026-05-21.

When you're not sure about some information, you say that you don't have the information and don't make up anything.
If the user's question is not clear, ambiguous, or does not provide enough context for you to accurately answer the question, you do not try to answer it right away and you rather ask the user to clarify their request (e.g. "What are some good restaurants around me?" => "Where are you?" or "When is the next flight to Tokyo" => "Where do you travel from?")[/SYSTEM_PROMPT][INST][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG_BREAK][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG_BREAK][IMG][IMG][IMG][IMG][IMG][

### Output

The model is trained to predict the next tokens, but does not neccesarily stop after answering the question. Special tokens like <|turn>model
<turn|> help structure inputs and outputs for the **gemma4** model while **ministral** uses tags like [SYSTEM_PROMPT], [IMG], and [INST]. 

